### Loading all CSVs into a SQLite database

In [2]:
import sqlite3
import pandas as pd
import os

DATA_DIR = "data"
DB_PATH = "olist.db"

csv_files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)

for table_name, filename in csv_files.items():
    path = os.path.join(DATA_DIR, filename)
    df = pd.read_csv(path)
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {df.shape}")

conn.close()
print("\nDatabase created at:", os.path.abspath(DB_PATH))


Loaded customers: (99441, 5)
Loaded orders: (99441, 8)
Loaded order_items: (112650, 7)
Loaded order_payments: (103886, 5)
Loaded order_reviews: (99224, 7)
Loaded products: (32951, 9)
Loaded sellers: (3095, 4)
Loaded geolocation: (1000163, 5)
Loaded category_translation: (71, 2)

Database created at: c:\Users\MSI\OneDrive - MOE Stem Schools\Desktop\Documents\GitHub\Robusta_Final_Data_Int\olist.db


### Verify all 9 tables loaded correctly

In [3]:
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
conn.close()

print("Tables in database:")
for t in tables:
    print(" -", t[0])

assert len(tables) == 9, "Expected 9 tables — check for errors above."
print("\nall 9 tables present.")


Tables in database:
 - customers
 - orders
 - order_items
 - order_payments
 - order_reviews
 - products
 - sellers
 - geolocation
 - category_translation

all 9 tables present.


### Setup — reusable query helper

In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("olist.db")

def q(sql):
    return pd.read_sql_query(sql, conn)


### — Basic SELECT + WHERE (filtering)


In [5]:
q("""
SELECT order_id, order_status, order_purchase_timestamp
FROM orders
WHERE order_status = 'delivered'
LIMIT 10;
""")


,order_id,order_status,order_purchase_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,2017-10-02 10:56:33
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,2018-07-24 20:41:37
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,2018-08-08 08:38:49
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,2017-11-18 19:28:06
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,2018-02-13 21:18:39
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,2017-07-09 21:57:05
6,6514b8ad8028c9f2cc2374ded245783f,delivered,2017-05-16 13:10:30
7,76c6e866289321a7c93b82b54852dc33,delivered,2017-01-23 18:29:09
8,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,2017-07-29 11:55:02
9,e6ce16cb79ec1d90b1da9085a6118aeb,delivered,2017-05-16 19:41:10


### JOIN for connecting tables


In [6]:
q("""
SELECT o.order_id, o.order_status, oi.product_id, oi.price
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
LIMIT 10;
""")


,order_id,order_status,product_id,price
0,e481f51cbdc54678b7cc49136f2d6af7,delivered,87285b34884572647811a353c7ac498a,29.99
1,53cdb2fc8bc7dce0b6741e2150273451,delivered,595fac2a385ac33a80bd5114aec74eb8,118.70
2,47770eb9100c2d0c44946d9cf07ec65d,delivered,aa4383b373c6aca5d8797843e5594415,159.90
3,949d5b44dbf5de918fe9c16f97b45f8a,delivered,d0b61bfb1de832b15ba9d266ca96e5b0,45.00
4,ad21c59c0840e6cb83a9ceb5573f8159,delivered,65266b2da20d04dbe00c5c2d3bb7859e,19.90
5,a4591c265e18cb1dcee52889e2d8acc3,delivered,060cb19345d90064d1015407193c233d,147.90
6,136cce7faa42fdb2cefd53fdc79a6098,invoiced,a1804276d9941ac0733cfd409f5206eb,49.90
7,6514b8ad8028c9f2cc2374ded245783f,delivered,4520766ec412348b8d4caa5e8a18c464,59.99
8,76c6e866289321a7c93b82b54852dc33,delivered,ac1789e492dcd698c5c10b97a671243a,19.90
9,e69bfb5eb88e0ed6a785585b27e16dbf,delivered,9a78fb9862b10749a117f7fc3c31f051,149.99


### GROUP BY + aggregates 
`GROUP BY` + `COUNT/SUM/AVG` 

In [7]:
q("""
SELECT customer_id, COUNT(order_id) AS total_orders
FROM orders
GROUP BY customer_id
ORDER BY total_orders DESC
LIMIT 10;
""")


,customer_id,total_orders
0,ffffe8b65bbe3087b653a978c870db99,1
1,ffffa3172527f765de70084a7e53aae8,1
2,ffff42319e9b2d713724ae527742af25,1
3,fffeda5b6d849fbd39689bb92087f431,1
4,fffecc9f79fd8c764f843e9951b11341,1
5,fffcb937e9dd47a13f05ecb8290f4d3e,1
6,fffc22669ca576ae3f654ea64c8f36be,1
7,fffb97495f78be80e2759335275df2aa,1
8,fffa0238b217e18a8adeeda0669923a3,1
9,fff93c1da78dafaaa304ff032abc6205,1


###  Multi-table JOIN + aggregate for real analytical questions


In [8]:
q("""
SELECT ct.product_category_name_english AS category, 
       COUNT(oi.order_item_id) AS items_sold,
       ROUND(SUM(oi.price), 2) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
JOIN category_translation ct ON p.product_category_name = ct.product_category_name
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 10;
""")

,category,items_sold,total_revenue
0,health_beauty,9670,1258681.34
1,watches_gifts,5991,1205005.68
2,bed_bath_table,11115,1036988.68
3,sports_leisure,8641,988048.97
4,computers_accessories,7827,911954.32
5,furniture_decor,8334,729762.49
6,cool_stuff,3796,635290.85
7,housewares,6964,632248.66
8,auto,4235,592720.11
9,garden_tools,4347,485256.46


###  Subquery / HAVING (finding repeat customers)
`HAVING` filters *after* aggregation (unlike `WHERE`, which filters before). Olist has no native reorder flag, so this is how you derive repeat-purchase signal via `customer_unique_id`.

In [9]:
q("""
SELECT customer_unique_id, COUNT(DISTINCT order_id) AS num_orders
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY customer_unique_id
HAVING num_orders > 1
ORDER BY num_orders DESC
LIMIT 10;
""")


,customer_unique_id,num_orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,ca77025e7201e3b30c44b472ff346268,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,1b6c7548a2a1f9037c1fd3ddfed95f33,7
5,f0e310a6839dce9de1638e0fe5ab282a,6
6,de34b16117594161a6a89c50b289d35a,6
7,dc813062e0fc23409cd255f7f53c7074,6
8,63cfc61cee11cbe306bff5857d00bfe4,6
9,47c1a3033b8b77b3ab6e109eb4d5fdf3,6


### Date functions
`julianday()` enables date math in SQLite 

In [10]:
q("""
SELECT customer_unique_id, 
       MIN(order_purchase_timestamp) AS first_order,
       MAX(order_purchase_timestamp) AS last_order,
       julianday(MAX(order_purchase_timestamp)) - julianday(MIN(order_purchase_timestamp)) AS days_active
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY customer_unique_id
ORDER BY days_active DESC
LIMIT 10;
""")


,customer_unique_id,first_order,last_order,days_active
0,32ea3bdedab835c3aa6cb68ce66565ef,2016-10-03 09:44:50,2018-06-28 11:46:04,633.084190
1,ccafc1c3f270410521c3c6f3b249870f,2016-10-06 19:33:34,2018-06-07 19:03:12,608.978912
2,d8f3c4f441a9b59a29f977df16724f38,2017-01-18 21:08:18,2018-08-24 17:52:59,582.864363
3,94e5ea5a8c1bf546db2739673060c43f,2016-10-05 21:10:56,2018-05-09 13:49:19,580.693322
4,87b3f231705783eb2217e25851c0a45d,2016-10-08 18:45:34,2018-05-04 11:14:37,572.686840
5,8f6ce2295bdbec03cd50e34b4bd7ba0a,2017-02-09 12:48:59,2018-07-31 22:03:59,537.385417
6,30b782a79466007756f170cb5bd6bbd8,2017-02-20 19:58:47,2018-07-30 22:09:45,525.090949
7,4e23e1826902ec9f208e8cc61329b494,2016-10-05 12:32:55,2018-03-13 22:28:21,524.413495
8,a1c61f8566347ec44ea37d22854634a1,2017-03-19 08:36:36,2018-08-25 11:01:56,524.100926
9,a262442e3ab89611b44877c7aaf77468,2017-02-17 18:27:58,2018-07-24 16:44:34,521.928194
